# ExtrusionExtraction: interactive Colab reconstruction (experimental)
Run the cells in your own Google runtime. CPU only; no paid upgrade or hosted backend. Free runtime availability is not guaranteed. The precompiled worker includes Java, ONNX and SkinTokens; model weights are downloaded separately and verified by the worker. References remain in your temporary runtime. Export before disconnecting. [Colab FAQ](https://research.google.com/colaboratory/faq.html).


In [ ]:
from pathlib import Path
import hashlib, json, os, re, subprocess, tarfile, tempfile, urllib.request
root = Path(tempfile.mkdtemp(prefix='ee-'))
# Optional exact four-part version, e.g. v0.1.2.3. Empty selects newest published compatible worker.
VERSION = ''
pattern = re.compile(r'ExtrusionExtraction-(v[0-9]+\.[0-9]+\.[0-9]+\.[0-9]+)-colab-linux-x86_64\.tar\.gz')
candidates = []
for page in range(1, 11):
    url = f'https://api.github.com/repos/HereLiesAz/ExtrusionExtraction/releases?per_page=100&page={page}'
    with urllib.request.urlopen(url, timeout=30) as response:
        releases = json.load(response)
    if not releases:
        break
    for release in releases:
        if release['draft']:
            continue
        for asset in release['assets']:
            match = pattern.fullmatch(asset['name'])
            if match and (not VERSION or match[1] == VERSION):
                candidates.append((tuple(map(int, match[1][1:].split('.'))), asset))
assert candidates, 'No published Colab worker found. A successful release must publish one first.'
version, asset = max(candidates, key=lambda entry: entry[0])
digest = asset.get('digest') or ''
assert re.fullmatch(r'sha256:[0-9a-f]{64}', digest), 'Release asset lacks SHA-256 metadata.'
assert 0 < asset['size'] <= 512 * 1024 * 1024, 'Invalid worker archive size.'
download_url = asset['browser_download_url']
assert download_url.startswith('https://github.com/HereLiesAz/ExtrusionExtraction/releases/download/'), 'Unexpected release URL.'
archive = root / 'worker.tar.gz'
sha = hashlib.sha256()
received = 0
with urllib.request.urlopen(download_url, timeout=60) as source, archive.open('wb') as target:
    while chunk := source.read(1024 * 1024):
        received += len(chunk)
        assert received <= asset['size'], 'Worker exceeds expected size.'
        sha.update(chunk)
        target.write(chunk)
assert received == asset['size'] and 'sha256:' + sha.hexdigest() == digest, 'Worker integrity check failed.'
# Fail closed if the runtime lacks safe tar extraction.
assert hasattr(tarfile, 'data_filter'), 'Safe tar extraction is unavailable in this runtime.'
with tarfile.open(archive, 'r:gz') as package:
    members = package.getmembers()
    assert all(m.name == 'worker' or m.name.startswith('worker/') for m in members)
    assert sum(m.size for m in members) <= 1024 * 1024 * 1024, 'Expanded worker exceeds size limit.'
    package.extractall(root, members=members, filter='data')
worker = root / 'worker'
archive.unlink()
print('Worker version:', '.'.join(map(str, version)))
print('Source revision:', (worker / 'source-revision.txt').read_text().strip())


## Runtime check
Checks the actual packaged Java, ONNX and SkinTokens runtime before uploading any references. No source build or Android SDK. Linux CPU and native ABI compatibility are required.


In [ ]:
subprocess.run([str(worker / 'run'), '--check'], check=True)


In [ ]:
# Paired-job lifecycle helpers; execution remains in the visible notebook cell.
import datetime, json, os, shutil, signal, subprocess, time

def report_pair(directory, code, state, detail):
    record = {'version': 1, 'pair': code, 'state': state,
              'reportedAt': datetime.datetime.now(datetime.timezone.utc).isoformat(),
              'detail': str(detail)[-240:]}
    temporary = directory / 'status.part'
    temporary.write_text(json.dumps(record), encoding='utf-8')
    temporary.replace(directory / 'status.json')

def cancellation_requested(directory, code):
    marker = directory / 'cancel.txt'
    return marker.is_file() and marker.stat().st_size == len('EECANCEL1:' + code) and marker.read_text(encoding='utf-8') == 'EECANCEL1:' + code

def publish_pair_result(directory, output, previous):
    bundle = output / 'reconstruction.eeresult'
    if not bundle.is_file():
        return previous
    stat = bundle.stat()
    identity = (stat.st_ino, stat.st_mtime_ns, stat.st_size)
    if identity != previous:
        temporary = directory / 'result.part'
        # Worker replaces its own completed bundle atomically; copying an opened inode
        # cannot mix bytes from two versions of that bundle.
        with bundle.open('rb') as source, temporary.open('wb') as target:
            shutil.copyfileobj(source, target)
        temporary.replace(directory / 'reconstruction.eeresult')
    return identity

def stop_owned_worker(process):
    if process.poll() is not None:
        return
    try:
        os.killpg(process.pid, signal.SIGTERM)
    except ProcessLookupError:
        return
    try:
        process.wait(timeout=5)
    except subprocess.TimeoutExpired:
        try:
            os.killpg(process.pid, signal.SIGKILL)
        except ProcessLookupError:
            pass
        process.wait()

def run_paired_job(directory, code, output, command):
    # Exclusive marker prevents notebook cell reruns from launching the same job again.
    with (directory / 'run-started').open('x'):
        pass
    if cancellation_requested(directory, code):
        report_pair(directory, code, 'CANCELLED', 'Cancelled before execution.')
        return 130
    report_pair(directory, code, 'RUNNING', 'Starting reconstruction worker.')
    log = directory / 'worker.log'
    process = None
    published = None
    latest = 'Starting reconstruction worker.'
    cancelled = False
    try:
        with log.open('wb') as writer, log.open('rb') as reader:
            process = subprocess.Popen(command, stdout=writer, stderr=subprocess.STDOUT,
                                       start_new_session=True)
            while True:
                chunk = reader.read(65536)
                if chunk:
                    text = chunk.decode('utf-8', errors='replace')
                    print(text, end='', flush=True)
                    lines = text.splitlines()
                    if lines:
                        latest = lines[-1][:240]
                published = publish_pair_result(directory, output, published)
                if process.poll() is not None:
                    # Drain output still buffered on disk, keeping notebook output visible.
                    for chunk in iter(lambda: reader.read(65536), b''):
                        print(chunk.decode('utf-8', errors='replace'), end='', flush=True)
                    break
                if cancellation_requested(directory, code):
                    cancelled = True
                    stop_owned_worker(process)
                    break
                report_pair(directory, code, 'RUNNING', latest)
                time.sleep(1)
        published = publish_pair_result(directory, output, published)
        status = process.returncode
        if cancelled:
            state, detail = 'CANCELLED', 'Cancellation confirmed; completed outputs retained.'
        elif status == 0 and published is not None:
            state, detail = 'SUCCEEDED', 'Reconstruction completed.'
        elif published is not None:
            state, detail = 'PARTIAL', 'Worker exited with errors; completed results retained.'
        else:
            state, detail = 'FAILED', 'No importable result; see notebook output.'
        report_pair(directory, code, state, detail)
        return 130 if cancelled else (status if status else (0 if published is not None else 1))
    except BaseException as exc:
        if process is not None:
            stop_owned_worker(process)
        try:
            publish_pair_result(directory, output, published)
        finally:
            report_pair(directory, code, 'CANCELLED' if isinstance(exc, KeyboardInterrupt) else 'FAILED',
                        'Notebook interrupted.' if isinstance(exc, KeyboardInterrupt) else 'Worker supervision failed; see notebook output.')
        raise


In [ ]:
# Optional API transfer: run this cell to pair the app with this notebook.
# Each pairing accepts one job. For manual upload, skip this cell.
import secrets
pair_code = secrets.token_hex(16)
pair_dir = Path('/content') / ('ee-transfer-' + pair_code)
pair_dir.mkdir(mode=0o700)
(pair_dir / 'pair.txt').write_text('EEPAIR1:' + pair_code, encoding='utf-8')
report_pair(pair_dir, pair_code, 'READY', 'Waiting for one app job.')
print('Pairing code:', pair_code)
print('In the app, authorize Google, choose this runtime, pair and send your job.')


In [ ]:
from google.colab import files
# Set True after sending an app job through the paired runtime; False for manual upload.
USE_API_TRANSFER = False
if USE_API_TRANSFER:
    pending = sorted(pair_dir.glob('job-*.eejob'))
    claimed = pair_dir / 'claimed.eejob'
    assert len(pending) == 1 and not claimed.exists(), 'Send exactly one job or create a fresh pairing.'
    incoming = pending[0]
    assert 16 <= incoming.stat().st_size <= 16 + 24 * 504 * 504 * 4, 'Invalid job size.'
    incoming.rename(claimed)
    if cancellation_requested(pair_dir, pair_code):
        report_pair(pair_dir, pair_code, 'CANCELLED', 'Cancelled before execution.')
        raise RuntimeError('Job cancelled. Create a new pairing for another job.')
    report_pair(pair_dir, pair_code, 'CLAIMED', 'Job received; waiting for reconstruction cell.')
    uploaded = {'reconstruction.eejob': claimed.read_bytes()}
else:
    upload_dir = Path(tempfile.mkdtemp(prefix='upload-', dir=root))
    previous = Path.cwd()
    try:
        os.chdir(upload_dir)
        uploaded = files.upload()
    finally:
        os.chdir(previous)
print(list(uploaded))


In [ ]:
import struct
ORDER = [] # only needed for individual images: exact filenames in viewing order
CELLS = 72
MASK = True
RIG = 'both' # none, template, learned, both; ignored for .eejob
refs = Path(tempfile.mkdtemp(prefix='refs-', dir=root))
output = root / ('output-' + refs.name)
command = [str(worker / 'run')]
jobs = [name for name in uploaded if name.lower().endswith('.eejob')]
if jobs:
    assert len(jobs) == 1 and len(uploaded) == 1, 'Upload exactly one job file.'
    data = uploaded[jobs[0]]
    assert len(data) >= 16, 'Truncated job.'
    magic, cells, flags, count = struct.unpack_from('>4I', data)
    assert magic == 0x45454a31 and 32 <= cells <= 96 and 0 <= flags <= 7 and 4 <= count <= 24
    assert len(data) == 16 + count * 504 * 504 * 4, 'Invalid job length.'
    path = refs / 'reconstruction.eejob'
    path.write_bytes(data)
    args = ['--job', str(root/'models'), str(output), str(path)]
    needs_native = bool(flags & 4)
    print(f'{count} references, {cells} cells, mask={bool(flags & 1)}, template={bool(flags & 2)}, learned={needs_native}')
else:
    assert 4 <= len(ORDER) <= 24 and len(set(ORDER)) == len(ORDER) and set(ORDER) == set(uploaded)
    assert 32 <= CELLS <= 96 and RIG in ('none', 'template', 'learned', 'both')
    paths = []
    for i, name in enumerate(ORDER):
        path = refs / f'{i:02d}.image'
        path.write_bytes(uploaded[name])
        paths.append(str(path))
    args = [str(root/'models'), str(output), str(CELLS), 'mask' if MASK else 'no-mask', RIG, *paths]
if USE_API_TRANSFER:
    status = run_paired_job(pair_dir, pair_code, output, command + args)
else:
    status = subprocess.run(command + args).returncode
print('Exit status:', status)
print('Successful exports:', [p.name for p in output.glob('*.glb')])
if status: print('At least one stage failed; inspect the log. Successful prior variants remain downloadable.')


In [ ]:
# Download the importable workspace result first, then completed standalone GLBs.
bundle = output / 'reconstruction.eeresult'
if bundle.is_file():
    files.download(str(bundle))
else:
    print('No importable result bundle. An older worker or a failed bundle write may have produced only GLBs.')
exports = sorted(output.glob('*.glb'))
assert exports, 'No completed GLBs.'
for export in exports:
    files.download(str(export))
